# What cuVS accepts

Run on Colab with a T4 GPU (Runtime, Change runtime type, T4 GPU): Colab comes with cuVS, so nothing heavy
is installed (on Kaggle, installing cuVS alone takes several minutes). Every step runs; none needs skipping, each
says when it starts, when it ends and what comes next, and the probe stops itself at its time budget.

Which metrics cuVS's brute force accepts (L1 and L-infinity in particular), and which data types (float32,
float16, int8, uint8) and metrics its IVF-Flat and CAGRA accept, each checked against NumPy on small data. About
3-5 minutes in all. Copy the output of the last step back.

In [ ]:
# Step 1 of 4: Which GPU (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 1 of 4: Which GPU, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(f"step 1 of 4: Which GPU: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 2 of 4, Install cuVS where missing", flush=True)

In [ ]:
# Step 2 of 4: Install cuVS where missing (about 1 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 2 of 4: Install cuVS where missing, about 1 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
# cuVS (and CuPy) for the CUDA version this machine's PyTorch was built with, only where missing: Colab comes with
# cuVS; Kaggle does not. Mixing CUDA 12 and 13 packages breaks NVIDIA's other libraries.
import subprocess
import sys

import torch

cuda = torch.version.cuda.split(".")[0]
print("PyTorch", torch.__version__, "for CUDA", torch.version.cuda)


def works(stmt):
    r = subprocess.run([sys.executable, "-c", stmt], check=False, capture_output=True, text=True)
    return r.returncode == 0


def install(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg, "--extra-index-url=https://pypi.nvidia.com"],
                       check=False, capture_output=True, text=True)
    print(pkg, "installed" if r.returncode == 0 else "not installed: " + (r.stderr.strip().splitlines() or ["?"])[-1])


if not works("import cupy"):
    install(f"cupy-cuda{cuda}x")
if not works("import h5py"):
    install("h5py")
# cuVS's Python package can be present without its library (libcuvs_c.so): reinstall both, for this CUDA
if not works("from cuvs.neighbors import brute_force, cagra, ivf_flat"):
    install(f"libcuvs-cu{cuda}")
    install(f"cuvs-cu{cuda}")
print("cuVS", "works" if works("from cuvs.neighbors import brute_force, cagra, ivf_flat") else "still does not work")
print(f"step 2 of 4: Install cuVS where missing: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 3 of 4, Write cuvs_probe.py", flush=True)

In [ ]:
# Step 3 of 4: Write cuvs_probe.py (seconds)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 3 of 4: Write cuvs_probe.py, seconds: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
source = "\"\"\"What NVIDIA cuVS offers siann, measured on an NVIDIA GPU (Colab or Kaggle, a T4 or newer).\n\n--capabilities: which metrics cuVS's brute force accepts, and which data types and metrics its IVF-Flat and\nCAGRA accept, each checked against exact search in NumPy on small Gaussian data (a metric's names tried in\nturn: cuVS spells L1 and L-infinity in several ways). A few seconds per case.\n\n--cagra: a CAGRA graph built by cuVS on SIFT1M (ann-benchmarks' file, downloaded; 32 links per vector pruned\nfrom 64, as siann's graphs), its own search's recall against ann-benchmarks' ground truth at a few search\nwidths, and the index saved in cuVS's file format (without the vectors) plus its links as a NumPy file, for\nsiann to read and to compare with its own graphs. A few minutes, most of it the download and the build.\n\nEvery line prints as it is measured, with its number out of the total and the minutes so far.\n\n    python cuvs_probe.py --capabilities\n    python cuvs_probe.py --cagra\n\"\"\"\nimport argparse\nimport os\nimport time\nimport urllib.request\n\nimport numpy as np\n\nSTART = time.perf_counter()\nSIFT_URL = \"http://ann-benchmarks.com/sift-128-euclidean.hdf5\"\n\n\nclass Deadline:\n    \"\"\"A hard limit on the whole run: work checks it before each step, and once it has passed, the step is\n    skipped with a line saying so (the results so far stand).\"\"\"\n\n    def __init__(self, seconds: float):\n        self.seconds, self.end = seconds, time.perf_counter() + seconds\n\n    def left(self) -> float:\n        return self.end - time.perf_counter()\n\n    def over(self, what: str) -> bool:\n        if time.perf_counter() < self.end:\n            return False\n        print(f\"  out of time (the {self.seconds:.0f} s budget): {what} skipped\", flush=True)\n        return True\n\n\nDEADLINE = Deadline(float(\"inf\"))\n\n\ndef elapsed():\n    return f\"{(time.perf_counter() - START) / 60:.1f} min so far\"\n\n\ndef distances(xb, xq, metric):\n    \"\"\"Every query's distance to every base vector (smaller is nearer: inner product and cosine negated).\"\"\"\n    if metric == \"l2\":\n        d = ((xq[:, None, :] - xb[None, :, :]) ** 2).sum(2)\n    elif metric == \"l1\":\n        d = np.abs(xq[:, None, :] - xb[None, :, :]).sum(2)\n    elif metric == \"linf\":\n        d = np.abs(xq[:, None, :] - xb[None, :, :]).max(2)\n    elif metric == \"ip\":\n        d = -(xq @ xb.T)\n    else:                                                  # cosine\n        d = -(xq / np.linalg.norm(xq, axis=1, keepdims=True)) @ (xb / np.linalg.norm(xb, axis=1, keepdims=True)).T\n    return d\n\n\ndef recall_tied(found, dist, k):\n    \"\"\"recall@k up to ties: the share of found neighbors no farther than each query's k-th nearest (byte data\n    has many tied distances, which any order may break).\"\"\"\n    found = np.asarray(found)\n    kth = np.sort(dist, axis=1)[:, k - 1:k]\n    ok = (found >= 0) & (found < dist.shape[1])\n    near = np.take_along_axis(dist, np.where(ok, found, 0), 1) <= kth + 1e-6 * np.abs(kth) + 1e-6\n    return float(np.mean(ok & near))\n\n\ndef recall(found, truth):\n    found = np.asarray(found)\n    return float(np.mean([len(set(f) & set(t)) / len(t) for f, t in zip(found, truth, strict=True)]))\n\n\ndef capabilities() -> None:\n    import cupy as cp\n    import cuvs\n    from cuvs.neighbors import brute_force, cagra, ivf_flat\n    print(f\"cuVS {getattr(cuvs, '__version__', '?')}, CuPy {cp.__version__}\", flush=True)\n    rng = np.random.default_rng(0)\n    n, d, nq, k = 5000, 32, 50, 10\n    xb = rng.standard_normal((n, d), dtype=np.float32)\n    xq = rng.standard_normal((nq, d), dtype=np.float32)\n    xbb = rng.integers(0, 256, (n, d)).astype(np.float32)   # integer data for the byte types\n    xqb = rng.integers(0, 256, (nq, d)).astype(np.float32)\n\n    # each metric's candidate names in cuVS\n    names = {\"l2\": (\"sqeuclidean\", \"euclidean\", \"l2\"), \"ip\": (\"inner_product\",), \"cosine\": (\"cosine\",),\n             \"l1\": (\"l1\", \"cityblock\", \"manhattan\"), \"linf\": (\"chebyshev\", \"linf\")}\n    cases = ([(\"brute force\", m, None) for m in names]\n             + [(\"IVF-Flat\", m, dt) for m in (\"l2\", \"ip\", \"cosine\", \"l1\", \"linf\")\n                for dt in (\"float32\", \"float16\", \"int8\", \"uint8\")]\n             + [(\"CAGRA\", m, dt) for m in (\"l2\", \"ip\", \"cosine\") for dt in (\"float32\", \"float16\", \"int8\", \"uint8\")])\n    print(f\"  {len(cases)} cases; recall@{k} up to ties against NumPy on {nq} queries, {n:,} x {d} (integer \"\n          f\"data for int8 and uint8); IVF-Flat probes every list, so it should be exact\", flush=True)\n    for i, (what, metric, dt) in enumerate(cases, 1):\n        if DEADLINE.over(f\"cases {i}-{len(cases)}\"):\n            return\n        if dt in (\"int8\", \"uint8\"):\n            base, qs = (xbb - 128 if dt == \"int8\" else xbb), (xqb - 128 if dt == \"int8\" else xqb)\n        else:\n            base, qs = xb, xq\n        dist = distances(base, qs, metric)\n        cast = {\"float32\": np.float32, \"float16\": np.float16, \"int8\": np.int8, \"uint8\": np.uint8, None: np.float32}[dt]\n        b, q = cp.asarray(base.astype(cast)), cp.asarray(qs.astype(cast))\n        label = f\"[{i}/{len(cases)}] {what}, {metric}\" + (f\", {dt}\" if dt else \"\")\n        results = []\n        for name in names[metric]:\n            try:\n                if what == \"brute force\":\n                    index = brute_force.build(b, metric=name)\n                    _, found = brute_force.search(index, q, k)\n                elif what == \"IVF-Flat\":\n                    index = ivf_flat.build(ivf_flat.IndexParams(n_lists=16, metric=name), b)\n                    _, found = ivf_flat.search(ivf_flat.SearchParams(n_probes=16), index, q, k)   # every list\n                else:\n                    index = cagra.build(cagra.IndexParams(metric=name, graph_degree=32,\n                                                          intermediate_graph_degree=64), b)\n                    _, found = cagra.search(cagra.SearchParams(itopk_size=128), index, q, k)\n                results.append(f\"'{name}': recall {recall_tied(cp.asnumpy(found), dist, k):.4f}\")\n                break                                       # the first name that works is enough\n            except Exception as e:  # noqa: BLE001 (not supported under this name: try the next, then report)\n                results.append(f\"'{name}': {type(e).__name__}: {str(e).splitlines()[0][:90] if str(e) else ''}\")\n        print(f\"  {label:34s} \" + \"; \".join(results) + f\"   ({elapsed()})\", flush=True)\n\n\ndef load_sift(path: str):\n    import h5py\n    if not os.path.exists(path):\n        print(f\"  downloading {SIFT_URL} (about 500 MB)\", flush=True)\n        t = time.perf_counter()\n        got, shown = 0, 0\n        with urllib.request.urlopen(SIFT_URL) as r, open(path + \".part\", \"wb\") as f:\n            total = int(r.headers.get(\"Content-Length\") or 0)\n            while chunk := r.read(8 << 20):\n                f.write(chunk)\n                got += len(chunk)\n                if got - shown >= 100 << 20:\n                    shown = got\n                    print(f\"    {got >> 20:,} of {total >> 20:,} MB ({elapsed()})\", flush=True)\n                if DEADLINE.over(\"the rest of the download, and the build\"):\n                    raise SystemExit(f\"  stopped: {got >> 20:,} of {total >> 20:,} MB downloaded within the budget\")\n        os.replace(path + \".part\", path)\n        print(f\"  downloaded in {time.perf_counter() - t:.0f} s ({elapsed()})\", flush=True)\n    with h5py.File(path, \"r\") as f:\n        return (np.asarray(f[\"train\"], dtype=np.float32), np.asarray(f[\"test\"], dtype=np.float32),\n                np.asarray(f[\"neighbors\"])[:, :10])\n\n\ndef cagra_export(path: str, out: str) -> None:\n    import cupy as cp\n    import cuvs\n    from cuvs.neighbors import cagra\n    steps = 5\n    print(f\"cuVS {getattr(cuvs, '__version__', '?')}: CAGRA on SIFT1M, {steps} steps\", flush=True)\n    xb, xq, gt = load_sift(path)\n    if DEADLINE.left() < 90:                                   # the build cannot be cut short: about a minute on a T4\n        raise SystemExit(f\"  stopped: {DEADLINE.left():.0f} s left of the budget, too little to build the graph\")\n    print(f\"  [1/{steps}] SIFT1M loaded: {xb.shape[0]:,} x {xb.shape[1]}, {len(xq):,} queries ({elapsed()})\", flush=True)\n    b, q = cp.asarray(xb), cp.asarray(xq)\n    params = cagra.IndexParams(metric=\"sqeuclidean\", graph_degree=32, intermediate_graph_degree=64)\n    t = time.perf_counter()\n    index = cagra.build(params, b)\n    cp.cuda.Device().synchronize()\n    build = time.perf_counter() - t\n    print(f\"  [2/{steps}] built: {build:.1f} s (graph degree 32, intermediate 64, cuVS's default build \"\n          f\"algorithm) ({elapsed()})\", flush=True)\n    print(f\"  [3/{steps}] cuVS's own search, 10,000 queries, recall@10 against ann-benchmarks' ground truth:\", flush=True)\n    for width in (32, 64, 128, 256):\n        if DEADLINE.over(f\"the search at itopk_size {width} and wider (the graph is still saved)\"):\n            break\n        sp = cagra.SearchParams(itopk_size=width)\n        cagra.search(sp, index, q, 10)                     # warm-up\n        cp.cuda.Device().synchronize()\n        t = time.perf_counter()\n        _, found = cagra.search(sp, index, q, 10)\n        cp.cuda.Device().synchronize()\n        sec = time.perf_counter() - t\n        print(f\"      itopk_size {width:>4}: recall {recall(cp.asnumpy(found), gt):.4f}, \"\n              f\"{len(xq) / sec:,.0f} queries/s ({elapsed()})\", flush=True)\n    os.makedirs(out, exist_ok=True)\n    fname = os.path.join(out, \"cagra_sift.cuvs\")\n    cagra.save(fname, index, include_dataset=False)\n    print(f\"  [4/{steps}] saved {fname}: {os.path.getsize(fname):,} bytes, cuVS's format without the vectors \"\n          f\"({elapsed()})\", flush=True)\n    graph = getattr(index, \"graph\", None)\n    if graph is None:\n        print(f\"  [5/{steps}] this cuVS version gives no access to the links (index.graph): only the cuVS file\",\n              flush=True)\n    else:\n        links = cp.asnumpy(cp.asarray(graph)).astype(np.int32)\n        np.save(os.path.join(out, \"cagra_sift_links.npy\"), links)\n        print(f\"  [5/{steps}] saved {os.path.join(out, 'cagra_sift_links.npy')}: links {links.shape}, \"\n              f\"{links.dtype} ({elapsed()})\", flush=True)\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--capabilities\", action=\"store_true\")\n    p.add_argument(\"--cagra\", action=\"store_true\")\n    p.add_argument(\"--sift\", default=\"sift-128-euclidean.hdf5\", help=\"where SIFT1M is, or is downloaded to\")\n    p.add_argument(\"--out\", default=\".\", help=\"where the CAGRA files are saved\")\n    p.add_argument(\"--budget\", type=float, default=180, help=\"seconds for the whole run; past it, the rest is skipped\")\n    a = p.parse_args()\n    global DEADLINE\n    DEADLINE = Deadline(a.budget)\n    if a.capabilities:\n        capabilities()\n    if a.cagra:\n        cagra_export(a.sift, a.out)\n\n\nif __name__ == \"__main__\":\n    main()\n"
with open('cuvs_probe.py', 'w', encoding='utf-8') as f:
    f.write(source)
print(f"wrote cuvs_probe.py: {len(source.splitlines())} lines")
print(f"step 3 of 4: Write cuvs_probe.py: done in {(__import__('time').time() - _t0) / 60:.1f} min; next: step 4 of 4, Capabilities", flush=True)

In [ ]:
# Step 4 of 4: Capabilities (about 2 min)
_t0 = __import__("time").time()
NB_START = globals().get("NB_START", _t0)   # the notebook's clock: 300 s in all
print(f"step 4 of 4: Capabilities, about 2 min: started, {(_t0 - NB_START) / 60:.1f} min into the notebook's 5", flush=True)
!python -u cuvs_probe.py --capabilities --budget {max(20, int(270 - (__import__('time').time() - NB_START)))}
print(f"step 4 of 4: Capabilities: done in {(__import__('time').time() - _t0) / 60:.1f} min; that was the last step", flush=True)